In [1]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset
ruta = '../data/raw/2020-Jan.csv'
df = pd.read_csv(ruta)

# 2. Diagnóstico
print(f"Total de filas: {df.shape[0]:,} | Total de columnas: {df.shape[1]}")
print("\n--- Primeras 5 filas ---")
display(df.head())

print("\n--- Tipos de datos y valores nulos ---")
resumen_nulos = pd.DataFrame({
    'Tipo': df.dtypes,
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2)
})
display(resumen_nulos)

print("\n--- Distribución de eventos ---")
display(df['event_type'].value_counts())

Total de filas: 4,264,752 | Total de columnas: 9

--- Primeras 5 filas ---


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-01-01 00:00:00 UTC,view,5809910,1602943681873052386,NaN,grattol,5.24,595414620,4adb70bb-edbd-4981-b60f-a05bfd32683a
1,2020-01-01 00:00:09 UTC,view,5812943,1487580012121948301,NaN,kinetics,3.97,595414640,c8c5205d-be43-4f1d-aa56-4828b8151c8a
2,2020-01-01 00:00:19 UTC,view,5798924,1783999068867920626,NaN,zinger,3.97,595412617,46a5010f-bd69-4fbe-a00d-bb17aa7b46f3
3,2020-01-01 00:00:24 UTC,view,5793052,1487580005754995573,NaN,NaN,4.92,420652863,546f6af3-a517-4752-a98b-80c4c5860711
4,2020-01-01 00:00:25 UTC,view,5899926,2115334439910245200,NaN,NaN,3.92,484071203,cff70ddf-529e-4b0c-a4fc-f43a749c0acb



--- Tipos de datos y valores nulos ---


,Tipo,Nulos,% Nulos
event_time,str,0,0.00
event_type,str,0,0.00
product_id,int64,0,0.00
category_id,int64,0,0.00
category_code,str,4190033,98.25
brand,str,1775630,41.64
price,float64,0,0.00
user_id,int64,0,0.00
user_session,str,1314,0.03



--- Distribución de eventos ---


event_type
view                2037608
cart                1148323
remove_from_cart     815024
purchase             263797
Name: count, dtype: int64

In [2]:
# 1. Limpieza básica y manejo de nulos
df_clean = df.dropna(subset=['user_session']).copy()
df_clean['brand'] = df_clean['brand'].fillna('Sin Marca')
df_clean['event_time'] = pd.to_datetime(df_clean['event_time'])

# Eliminamos precios negativos o en 0 
df_clean = df_clean[df_clean['price'] > 0]

print("Distribución de eventos limpios:")
print(df_clean['event_type'].value_counts())

# 2. Tabla de Embudo por Marca (15 marcas con más tráfico)
top_marcas = df_clean['brand'].value_counts().head(15).index
funnel_marcas = df_clean[df_clean['brand'].isin(top_marcas)].pivot_table(
    index='brand', 
    columns='event_type', 
    values='user_session', 
    aggfunc='count', 
    fill_value=0
).reset_index()

# Calculamos tasas de conversión
funnel_marcas['tasa_carrito_%'] = (funnel_marcas['cart'] / funnel_marcas['view'] * 100).round(2)
funnel_marcas['tasa_compra_%'] = (funnel_marcas['purchase'] / funnel_marcas['view'] * 100).round(2)
display(funnel_marcas.sort_values(by='purchase', ascending=False))

# 3. Segmentación RFM (Solo sobre eventos de compra 'purchase')
compras = df_clean[df_clean['event_type'] == 'purchase'].copy()
fecha_referencia = compras['event_time'].max() + pd.Timedelta(days=1)

rfm = compras.groupby('user_id').agg({
    'event_time': lambda x: (fecha_referencia - x.max()).days, # Recencia
    'user_session': 'nunique',                                 # Frecuencia (órdenes únicas)
    'price': 'sum'                                             # Monto total gastado
}).reset_index()

rfm.columns = ['user_id', 'recencia_dias', 'frecuencia', 'monto_total']

# Clasificación simple de segmentos de negocio
def segmentar_cliente(row):
    if row['frecuencia'] >= 3 and row['monto_total'] >= rfm['monto_total'].quantile(0.75):
        return 'VIP'
    elif row['frecuencia'] >= 2 and row['recencia_dias'] <= 15:
        return 'Leal'
    elif row['frecuencia'] >= 2 and row['recencia_dias'] > 15:
        return 'En Riesgo'
    else:
        return 'Ocasional / Nueva'

rfm['segmento'] = rfm.apply(segmentar_cliente, axis=1)
print("\nDistribución de Segmentos de Clientas:")
print(rfm['segmento'].value_counts())

# 4. Exportar tablas procesadas para Power BI y SQL
funnel_marcas.to_csv('../data/processed/funnel_por_marca.csv', index=False)
rfm.to_csv('../data/processed/segmentacion_rfm.csv', index=False)
compras.to_csv('../data/processed/ventas_limpias.csv', index=False)
print("\n¡Archivos exportados con éxito en data/processed/!")

Distribución de eventos limpios:
event_type
view                2032175
cart                1145588
remove_from_cart     814254
purchase             263760
Name: count, dtype: int64


event_type,brand,cart,purchase,remove_from_cart,view,tasa_carrito_%,tasa_compra_%
0,Sin Marca,485191,111676,350079,820568,59.13,13.61
13,runail,117867,26596,79385,144125,81.78,18.45
8,irisk,68389,14827,40365,87235,78.40,17.00
12,masura,58625,12234,55166,77496,75.65,15.79
6,grattol,48649,11828,35603,97695,49.80,12.11
7,ingarden,21038,5156,17519,34675,60.67,14.87
4,estel,19472,4504,11273,46752,41.65,9.63
2,bpw.style,15393,4175,13375,22566,68.21,18.50
14,uno,14199,3811,9301,26302,53.98,14.49
9,italwax,12867,3455,6720,21986,58.52,15.71



Distribución de Segmentos de Clientas:
segmento
Ocasional / Nueva    25018
Leal                  2040
En Riesgo              611
VIP                    551
Name: count, dtype: int64

¡Archivos exportados con éxito en data/processed/!
